# Data Challenge : Lynred data

---

In [ ]:
%load_ext autoreload
%autoreload 2

In [2]:
import pandas as pd
import numpy as np
import json
import cv2
import glob, os
import matplotlib.pyplot as plt
from skimage import io
from scipy.ndimage import median_filter
from scipy.ndimage import convolve1d
from functions import *
import seaborn as sns

---

## Loading the data

In [ ]:
def load_images(folder='train',type='VGA',sequence='sequence_1', dyn='low dyn with columns 1', force_gray=False):
    """
    function that load a folder of images

    args : 
    folder : folder type
    type : type of the frame (HD, VGA, SXGA)
    sequence : sequence_1, sequence_2, sequence_3
    dyn : low dyn with columns 1, low dyn with columns 2, low dyn with columns 3
    force_gray : bool to load in gray (one array)

    return :
    list of the images of the folder
    """
    images = []

    chemin_recherche = os.path.join(folder, type, sequence, dyn, '*.png')

    fichiers_trouves = sorted(glob.glob(chemin_recherche))

    if force_gray == True:
        for image_path in fichiers_trouves:
            img = io.imread(image_path, as_gray=True)
            images.append(img)
    else:
        for image_path in fichiers_trouves:
            img = io.imread(image_path)
            images.append(img)
            
    return images

In [ ]:
def load_dataset(folder='train'):
    """
    function that load all the data set paths

    return : 
    dict : with every concept containing image paths
    list : with every image path
    """
    types = ['HD', 'SXGA', 'VGA']
    sequences = ['sequence_1', 'sequence_2', 'sequence_3']
    dynamics = [
        'high_dyn',
        'low dyn',
        'low dyn with columns 1',
        'low dyn with columns 2',
        'low dyn with columns 3'
    ]

    dataset_dict = {}
    all_images_list = []

    for t in types:
        dataset_dict[t] = {}
        for seq in sequences:
            dataset_dict[t][seq] = {}
            for dyn in dynamics:

                # Récupère que les fichiers png du dossier
                path_pattern = f"{folder}/{t}/{seq}/{dyn}/*.png"
                
                paths_in_folder = glob.glob(path_pattern)
                
                dataset_dict[t][seq][dyn] = paths_in_folder
                all_images_list.extend(paths_in_folder)

    return dataset_dict, all_images_list

In [ ]:
def convert_to_8_img(img_16bit):
    """""
    Convert 16 bits img into 8 bits using percentiles to not be biased by outliers  
    arg : img 
    return : converted img in 8 bits
    """""
    # 1. Utiliser des percentiles pour ignorer les pixels aberrants (très chauds ou très froids)
    # Cela permet de mieux voir les détails de la scène sans être ébloui par les extrêmes
    p2, p98 = np.percentile(img_16bit, (2, 98))
    img_clipped = np.clip(img_16bit, p2, p98)
    
    # 2. Normaliser entre 0 et 255
    img_8bit = ((img_clipped - p2) / (p98 - p2) * 255).astype(np.uint8)
    return img_8bit

In [ ]:
minus_datetrain = load_images()

## Explore the Dataset

In [ ]:
fig, axes = plt.subplots(2, 5, figsize=(10,5))

for i in range(0,2,1):
    for j in range(0,5,1):
        index_image = i * 5 + j 
        axes[i, j].imshow(minus_datetrain[index_image], cmap='gray')
        axes[i, j].set_title(f'Frame {index_image}')
        axes[i, j].axis('off')

In [ ]:
fig, axes = plt.subplots(2, 5, figsize=(10,5))

for i in range(0,2,1):
    for j in range(0,5,1):
        index_image = i * 5 + j 
        axes[i, j].imshow(convert_to_8_img(minus_datetrain[index_image]), cmap='gray')
        axes[i, j].set_title(f'Frame {index_image}')
        axes[i, j].axis('off')

---
## First find defects columns by using the variance

In [ ]:
def metrique_img(image, mean_not_var = True):
    """
    Fonction pour discriminer avec la variance avec un threshold global
    """
    hauteur, longueur = image.shape
    if mean_not_var:
        var = np.mean(image, axis=0)
        label = 'Moyenne'
    else:
        var = np.var(image, axis=0)
        label = 'Variance'
    threshold = np.mean(var) + 2*np.std(var)
    defect_columns = np.where(var > threshold)[0]
    print(f"Seuil fixé à : {threshold:.2f}")
    print(f"Nombre de colonnes dépassant le seuil : {len(defect_columns)}")
    print(f"Index de ces colonnes : {defect_columns}")

    plt.figure(figsize=(12, 6))
    
    # Tracer la courbe de toutes les variances/moyennes
    plt.plot(var, label=label, color='#1f77b4', linewidth=1.5)
    
    # Tracer la ligne du threshold (seuil)
    plt.axhline(y=threshold, color='red', linestyle='--', linewidth=2, label=f'Threshold ({threshold:.2f})')
    
    # Mettre un point rouge sur le graphe pour chaque "pire" colonne
    plt.scatter(defect_columns, var[defect_columns], color='red', zorder=5, label='Colonnes critiques')

    # Personnalisation du graphe
    plt.title(f'{label} des intensités de pixels par colonne', fontsize=14)
    plt.xlabel("Index de la colonne (Position X dans l'image)", fontsize=12)
    plt.ylabel(f'{label}', fontsize=12)
    plt.legend()
    plt.grid(True, linestyle=':', alpha=0.7)

metrique_img(minus_datetrain[0])
metrique_img(minus_datetrain[0], mean_not_var=False)

> Just to know we have a noisy columns in Img[0] at x = 395 AND a blinking one at x = 450.

On observe que les bonnes détections faites sont réussies par la moyenne, mais notre threshold est global donc ne s'adapte pas à l'évolution dans l'image.

In [ ]:
def metrique_img_local(image, mean_not_var=True, taille_fenetre=50):
    """
    Fonction pour discriminer les colonnes anormales avec un threshold LOCAL adaptatif.
    """
    if mean_not_var:
        metrique = np.mean(image, axis=0)
        label = 'Moyenne'
    else:
        metrique = np.var(image, axis=0)
        label = 'Variance'
        
    # 2. Calcul du Seuil Local
    # a) Calcul de la tendance locale (moyenne glissante)
    filtre = np.ones(taille_fenetre) / taille_fenetre
    tendance_locale = convolve1d(metrique, filtre, mode='reflect')
    
    # b) Marge de tolérance
    marge_tolerance = np.std(metrique)
    
    # Le seuil n'est plus un simple nombre, c'est un tableau de la même taille que l'image !
    threshold_local = tendance_locale + marge_tolerance
    
    # 3. Détection des anomalies
    # On compare chaque colonne avec SON seuil local
    defect_columns = np.where(metrique > threshold_local)[0]
    
    print(f"Analyse basée sur : {label}")
    print(f"Marge de tolérance (+ 2*std) : {marge_tolerance:.2f}")
    print(f"Nombre de colonnes dépassant le seuil local : {len(defect_columns)}")
    print(f"Index de ces colonnes : {defect_columns}")

    # 4. Affichage du graphe
    plt.figure(figsize=(12, 6))
    
    # Tracer la courbe de toutes les variances/moyennes
    # Correction : label=label (sans les guillemets)
    plt.plot(metrique, label=label, color='#1f77b4', linewidth=1.5, alpha=0.8)
    
    # Tracer la courbe du threshold local
    # On utilise plt.plot au lieu de axhline car le seuil n'est plus une ligne droite horizontale
    plt.plot(threshold_local, color='orange', linestyle='--', linewidth=2.5, label='Seuil Local Adaptatif')
    
    # Mettre un point rouge sur le graphe pour chaque "pire" colonne
    plt.scatter(defect_columns, metrique[defect_columns], color='red', zorder=5, label='Colonnes critiques')

    # Personnalisation du graphe
    plt.title(f'{label} des intensités par colonne (avec Seuil Local)', fontsize=14)
    plt.xlabel('Index de la colonne (Position X dans l\'image)', fontsize=12)
    plt.ylabel(f'{label}', fontsize=12)
    plt.legend()
    plt.grid(True, linestyle=':', alpha=0.7)
    
    plt.tight_layout()
    plt.show()

metrique_img_local(minus_datetrain[0], mean_not_var=True)
print(50*'-')
metrique_img_local(minus_datetrain[0], mean_not_var=False)

On observe bien que la méthode de la variance est pas du tout pertinente.

## Testons sur d'autres images

In [ ]:
metrique_img(minus_datetrain[3], mean_not_var=True)
print(50*'-')
metrique_img_local(minus_datetrain[3], mean_not_var=True)

Comme on peut le voir on doit appliquer dans les deux sens (logique).

In [ ]:
def mean_threshold_local(image, taille_fenetre=50, visual_graph = False):

    height = image.shape[0]

    metrique = np.mean(image, axis=0)
    label = 'Moyenne'

    filtre = np.ones(taille_fenetre) / taille_fenetre
    tendance_locale = convolve1d(metrique, filtre, mode='reflect')
    
    #Marge de tolérance
    marge_tolerance = np.std(metrique)
    
    # Le seuil n'est plus un simple nombre, c'est un tableau de la même taille que l'image !
    threshold_haut = tendance_locale + marge_tolerance
    threshold_bas = tendance_locale - marge_tolerance
    
    # Détection bilatérale avec l'écart absolu (np.abs)
    defect_columns = np.where(np.abs(metrique - tendance_locale) > marge_tolerance)[0]

    predit_dict = {}
    for col in defect_columns:
        # On convertit 'col' en int natif Python pour éviter les soucis avec JSON/Dictionnaires
        # Et on indique que le défaut s'étend de la ligne 0 jusqu'en bas (height)
        predit_dict[int(col)] = [(0, height)]

    if visual_graph:
        # Printing results    
        print(f"Analyse basée sur : {label}")
        print(f"Marge de tolérance (+ 2*std) : {marge_tolerance:.2f}")
        print(f"Nombre de colonnes dépassant le seuil local : {len(defect_columns)}")
        print(f"Index de ces colonnes : {defect_columns}")

        # Affichage du graphe
        plt.figure(figsize=(12, 6))
        
        # Tracer la courbe de toutes les variances/moyennes
        plt.plot(metrique, label=label, color='#1f77b4', linewidth=1.5, alpha=0.8)
        
        # Tracer la courbe du threshold local
        plt.plot(threshold_haut, color='orange', linestyle='--', linewidth=2, label='Seuil Haut (+2 std)')
        plt.plot(threshold_bas, color='green', linestyle='--', linewidth=2, label='Seuil Bas (-2 std)')
        
        # Mettre un point rouge sur le graphe pour chaque "pire" colonne
        plt.scatter(defect_columns, metrique[defect_columns], color='red', zorder=5, label='Colonnes critiques')

        # Personnalisation du graphe
        plt.title(f'{label} des intensités par colonne (avec Seuil Local)', fontsize=14)
        plt.xlabel('Index de la colonne (Position X dans l\'image)', fontsize=12)
        plt.ylabel(f'{label}', fontsize=12)
        plt.legend()
        plt.grid(True, linestyle=':', alpha=0.7)
        
        plt.tight_layout()
        plt.show()
    return predit_dict

mean_threshold_local(minus_datetrain[3])

In [ ]:
for img in minus_datetrain:
    mean_threshold_local(img)

Ici ça marche vrmt super mais c'est un peu ce qu'on attends d'erreurs grossières comme ça, j'en profite pour regarder avec 

In [ ]:
def mean_threshold_local_band(img, split=8):
    

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.ndimage import convolve1d

def mean_threshold_local_band(image, taille_fenetre=50, nb_bandes=4, visual_graph=False):
    height = image.shape[0]
    
    # Calcul de la hauteur de chaque bande
    band_height = height // nb_bandes
    
    predit_dict = {}
    
    # Préparation de l'affichage si demandé
    if visual_graph:
        # Crée un sous-graphe pour chaque bande, empilés verticalement
        fig, axes = plt.subplots(nb_bandes, 1, figsize=(12, 3.5 * nb_bandes), sharex=True)
        if nb_bandes == 1:
            axes = [axes] # Sécurité si nb_bandes = 1 pour garder l'itération possible

    for b in range(nb_bandes):
        # 1. Définir les limites Y de la bande actuelle
        y_start = b * band_height
        # La dernière bande prend tous les pixels restants (pour éviter les soucis de division)
        y_end = height if b == nb_bandes - 1 else (b + 1) * band_height
        
        # 2. Extraire la sous-image (la bande)
        band_image = image[y_start:y_end, :]
        
        # --- VOTRE LOGIQUE DE DÉTECTION ---
        metrique = np.mean(band_image, axis=0)
        
        filtre = np.ones(taille_fenetre) / taille_fenetre
        tendance_locale = convolve1d(metrique, filtre, mode='reflect')
        
        # Marge de tolérance (Note : votre print indiquait 2*std, mais le code fait 1*std. J'ai gardé votre code tel quel)
        marge_tolerance = np.std(metrique)
        
        threshold_haut = tendance_locale + marge_tolerance
        threshold_bas = tendance_locale - marge_tolerance
        
        defect_columns = np.where(np.abs(metrique - tendance_locale) > marge_tolerance)[0]
        
        # 3. Enregistrement des résultats pour la bande en cours
        for col in defect_columns:
            col_int = int(col)
            # Si la colonne n'est pas encore dans le dictionnaire, on l'initialise
            if col_int not in predit_dict:
                predit_dict[col_int] = []
            
            # On ajoute le segment (y_start, y_end) pour indiquer OÙ la colonne est défectueuse
            predit_dict[col_int].append((y_start, y_end))

        # 4. Remplissage du graphique pour cette bande spécifique
        if visual_graph:
            ax = axes[b]
            ax.plot(metrique, label=f'Moyenne', color='#1f77b4', linewidth=1.5, alpha=0.8)
            ax.plot(threshold_haut, color='orange', linestyle='--', linewidth=2, label='Seuil Haut (+1 std)')
            ax.plot(threshold_bas, color='green', linestyle='--', linewidth=2, label='Seuil Bas (-1 std)')
            ax.scatter(defect_columns, metrique[defect_columns], color='red', zorder=5, label='Colonnes critiques')

            ax.set_title(f'Bande {b+1} (Lignes {y_start} à {y_end}) | {len(defect_columns)} défauts', fontsize=12)
            ax.set_ylabel('Moyenne')
            ax.grid(True, linestyle=':', alpha=0.7)
            
            # N'afficher la légende que sur le premier graphe pour ne pas surcharger
            if b == 0:
                ax.legend(loc='upper right')

    if visual_graph:
        # On met le label X seulement sur le graphe tout en bas
        axes[-1].set_xlabel('Index de la colonne (Position X dans l\'image)', fontsize=12)
        plt.tight_layout()
        plt.show()

    return predit_dict

# Appel de la fonction :
resultats = mean_threshold_local_band(minus_datetrain[3], taille_fenetre=50, nb_bandes=4, visual_graph=True)

---

# Fragmented columns

## Using Bands

---

## Try to detect blinky noisy columns (not fragmented)

> L'idée est de voir les les pixels comme ayant 3 valeurs (x,y,t), t étant le temps. 

On voudrait supprimer le reste de l'image soit pas soustraction, soit pas dérivation pour garder que les différences un peu particulières des images au cours du temps.

---

# Création d'une fonction d'évaluation de nos méthodes de détections

In [ ]:
print(get_defect_coordinates(load_json("results/VGA_sequence_1_config_1.json"), 0))

In [ ]:
print(get_defect_coordinates(load_json("results/VGA_sequence_1_config_3.json"), 0))

In [ ]:
def evaluate_detection(vrai_dict, predit_dict, printing = False):
    """
    Évalue la détection avec correspondance exacte (sans tolérance).
    Force la conversion des clés en entiers (int) pour éviter les erreurs de type string/int.
    """
    vrais_positifs = 0
    faux_negatifs = 0
    faux_positifs = 0
    
    # On convertit toutes les colonnes prédites en int pour être sûr du format
    colonnes_predites_int = {int(x) for x in predit_dict.keys()}
    colonnes_predites_utilisees = set()

    # =========================================================
    # 1. Vérifier ce qui est bien détecté et ce qui est oublié
    # =========================================================
    for x_vrai_raw, infos_vrai in vrai_dict.items():
        x_vrai = int(x_vrai_raw) # On force la vérité terrain en int
        
        if x_vrai in colonnes_predites_int:
            vrais_positifs += 1
            colonnes_predites_utilisees.add(x_vrai)
        else:
            faux_negatifs += 1

    # =========================================================
    # 2. Vérifier les fausses alarmes (Faux Positifs)
    # =========================================================
    for x_pred in colonnes_predites_int:
        # Si la colonne prédite n'a pas matché avec une vraie colonne
        if x_pred not in colonnes_predites_utilisees:
            faux_positifs += 1

    # =========================================================
    # 3. Calcul des statistiques finales
    # =========================================================
    precision = vrais_positifs / (vrais_positifs + faux_positifs) if (vrais_positifs + faux_positifs) > 0 else 0
    rappel = vrais_positifs / (vrais_positifs + faux_negatifs) if (vrais_positifs + faux_negatifs) > 0 else 0
    if (precision + rappel) > 0:
        f1_score = 2 * (precision * rappel) / (precision + rappel)
    else:
        f1_score = 0

    if printing:
        print("\n--- RÉSULTATS DE LA DÉTECTION ---")
        print(f"Vrais Positifs (Bien trouvés)  : {vrais_positifs}")
        print(f"Faux Négatifs (Oubliés)        : {faux_negatifs}")
        print(f"Faux Positifs (Fausses alarmes): {faux_positifs}")
        print(f"Précision (Fiabilité)          : {precision*100:.1f}%")
        print(f"Rappel (Taux de découverte)    : {rappel*100:.1f}%")
        print(f"F1-Score (Score global)        : {f1_score*100:.1f}%")
    
    return vrais_positifs, faux_positifs, faux_negatifs, f1_score

In [ ]:
evaluate_detection(get_defect_coordinates(load_json('results/VGA_sequence_1_config_1.json'), 0), detecter_et_mesurer_defauts_complet(minus_datetrain[0]), printing=True)

In [ ]:
evaluate_detection(get_defect_coordinates(load_json('results/VGA_sequence_1_config_1.json'), 0), mean_threshold_local(minus_datetrain[0]), printing=True)

La méthode d'Aurane marche pas du tout... (par rapport à un calcul du mean)

### Généralisons à toutes notre data_train !!

In [ ]:
def evaluate_detection(vrai_dict, predit_dict, printing = False):
    """
    Évalue la détection avec correspondance exacte (sans tolérance).
    Force la conversion des clés en entiers (int) pour éviter les erreurs de type string/int.
    Calcule et retourne la Précision, le Rappel et le F1-Score.
    """
    vrais_positifs = 0
    faux_negatifs = 0
    faux_positifs = 0
    
    # On convertit toutes les colonnes prédites en int pour être sûr du format
    colonnes_predites_int = {int(x) for x in predit_dict.keys()}
    colonnes_predites_utilisees = set()

    # =========================================================
    # 1. Vérifier ce qui est bien détecté et ce qui est oublié
    # =========================================================
    for x_vrai_raw, infos_vrai in vrai_dict.items():
        x_vrai = int(x_vrai_raw) # On force la vérité terrain en int
        
        # On vérifie la correspondance exacte
        if x_vrai in colonnes_predites_int:
            vrais_positifs += 1
            colonnes_predites_utilisees.add(x_vrai)
        else:
            faux_negatifs += 1

    # =========================================================
    # 2. Vérifier les fausses alarmes (Faux Positifs)
    # =========================================================
    for x_pred in colonnes_predites_int:
        # Si la colonne prédite n'a pas matché avec une vraie colonne
        if x_pred not in colonnes_predites_utilisees:
            faux_positifs += 1

    # =========================================================
    # 3. Calcul des statistiques finales
    # =========================================================
    precision = vrais_positifs / (vrais_positifs + faux_positifs) if (vrais_positifs + faux_positifs) > 0 else 0
    rappel = vrais_positifs / (vrais_positifs + faux_negatifs) if (vrais_positifs + faux_negatifs) > 0 else 0
    
    # Calcul du F1-Score
    if (precision + rappel) > 0:
        f1_score = 2 * (precision * rappel) / (precision + rappel)
    else:
        f1_score = 0
    
    if printing:
        print("\n--- RÉSULTATS DE LA DÉTECTION (Correspondance exacte) ---")
        print(f"Vrais Positifs (Bien trouvés)  : {vrais_positifs}")
        print(f"Faux Négatifs (Oubliés)        : {faux_negatifs}")
        print(f"Faux Positifs (Fausses alarmes): {faux_positifs}")
        print(f"Précision (Fiabilité)          : {precision*100:.1f}%")
        print(f"Rappel (Taux de découverte)    : {rappel*100:.1f}%")
        print(f"F1-Score (Score global)        : {f1_score*100:.1f}%")

    return vrais_positifs, faux_positifs, faux_negatifs, f1_score

In [ ]:
evaluate_detection(get_defect_coordinates(load_json('results/VGA_sequence_1_config_1.json'), 0), mean_threshold_local(minus_datetrain[0]), printing=True)

In [ ]:
def evaluate_sequence_detection(methode_detection=mean_threshold_local, 
                                folder='train', img_type='VGA', sequence='sequence_1', dyn='low dyn with columns 1'):
    """
    Évalue la détection sur toute une séquence d'images.
    dataset_name : Le nom du dataset (qui servira à trouver les images et le JSON)
    methode_detection : La fonction à utiliser pour prédire les défauts
    """
    chiffre = int(dyn.split()[-1])

    data = load_images(folder=folder, type=img_type, sequence=sequence, dyn=dyn, force_gray=False)
    json_name = f'{img_type}_{sequence}_config_{chiffre}'
    
    # 2. Charger le JSON une seule fois pour toute la séquence !
    # (J'ajoute .json à la fin, ajuste selon ton nommage réel)
    chemin_json = f'results/{json_name}.json'
    json_data = load_json(chemin_json)
    
    # Initialisation des compteurs GLOBAUX
    total_vp = 0
    total_fp = 0
    total_fn = 0

    # 3. Boucle sur chaque image
    for i in range(len(data)):
        # Vérité terrain pour l'image 'i'
        vrai_dict = get_defect_coordinates(json_data, i)
        
        # Prédiction de ton algorithme pour l'image 'i'
        predit_dict = methode_detection(data[i])
        
        # On récupère les valeurs de retour de notre fonction evaluate_detection
        # (Astuce : mets les prints de evaluate_detection en commentaire si tu ne 
        # veux pas polluer ton terminal avec les détails de chaque image)
        vp, fp, fn, f1_image = evaluate_detection(vrai_dict, predit_dict)
        
        # On ajoute aux compteurs globaux
        total_vp += vp
        total_fp += fp
        total_fn += fn

    # =========================================================
    # 4. Calcul des scores globaux de la séquence
    # =========================================================
    precision_seq = total_vp / (total_vp + total_fp) if (total_vp + total_fp) > 0 else 0
    rappel_seq = total_vp / (total_vp + total_fn) if (total_vp + total_fn) > 0 else 0
    
    if (precision_seq + rappel_seq) > 0:
        f1_score_seq = 2 * (precision_seq * rappel_seq) / (precision_seq + rappel_seq)
    else:
        f1_score_seq = 0

    # Affichage des résultats
    print("\n" + "="*50)
    print(f"RÉSULTATS GLOBAUX - SÉQUENCE : type : {img_type} sequence : {sequence} dyn : {dyn}")
    print("="*50)
    print(f"Total Vrais Positifs (VP) : {total_vp}")
    print(f"Total Faux Négatifs (FN)  : {total_fn} (Oublis)")
    print(f"Total Faux Positifs (FP)  : {total_fp} (Fausses alarmes)")
    print("-" * 50)
    print(f"Précision de la séquence  = {precision_seq*100:.2f}%")
    print(f"Recall de la séquence     = {rappel_seq*100:.2f}%")
    print(f"F1_score de la séquence   = {f1_score_seq*100:.2f}%")
    print("="*50 + "\n")
    
    largeur_image = data[0].shape[1] 
    total_colonnes_evaluees = len(data) * largeur_image
        
    # Les Vrais Négatifs (TN) : Les colonnes saines correctement ignorées
    total_tn = total_colonnes_evaluees - (total_vp + total_fp + total_fn)
        
    # Format de la matrice : [[TN, FP], [FN, TP]]
    conf = [[total_tn, total_fp],[total_fn, total_vp]]
    
    return precision_seq, rappel_seq, f1_score_seq

# --- Exemple d'utilisation ---
'''
evaluate_sequence_detection(
    methode_detection=mean_threshold_local,
    folder="train",
    img_type="HD",
    sequence="sequence_3",
    dyn="low dyn with columns 3"
)
print(50*"==")
evaluate_sequence_detection(
    methode_detection=mean_threshold_local_band,
    folder="train",
    img_type="HD",
    sequence="sequence_3",
    dyn="low dyn with columns 3"
)
print(50*"==")
evaluate_sequence_detection(
    methode_detection=detecter_et_mesurer_defauts_complet,
    folder="train",
    img_type="HD",
    sequence="sequence_3",
    dyn="low dyn with columns 3"
)
'''